In [0]:
-- ============================================================
-- SECTION 1: ROW COUNT VALIDATION
-- ============================================================

-- Check 1: Customers Bronze Row Count → 
SELECT COUNT(*) AS customers_bronze_count
FROM retail_sales_catalog.bronze.customers_raw
union all
SELECT COUNT(*) AS products_bronze_count
FROM retail_sales_catalog.bronze.products_raw
union all
-- Check 3: Stores Bronze Row Count → 
SELECT COUNT(*) AS stores_bronze_count
FROM retail_sales_catalog.bronze.stores_raw
union all
-- Check 4: Sales Bronze Row Count →
SELECT COUNT(*) AS sales_bronze_count
FROM retail_sales_catalog.bronze.sales_raw;

-- Check 5: Silver Layer Row Count Summary
SELECT 'DimCustomer' AS table_name, COUNT(*) AS row_count FROM retail_sales_catalog.silver.DimCustomer
UNION ALL SELECT 'DimProduct',  COUNT(*) FROM retail_sales_catalog.silver.DimProduct
UNION ALL SELECT 'DimStore',    COUNT(*) FROM retail_sales_catalog.silver.DimStore
UNION ALL SELECT 'FactSales',   COUNT(*) FROM retail_sales_catalog.silver.FactSales;

-- ============================================================
-- SECTION 2: DIMCUSTOMER TRANSFORMATION VALIDATION
-- ============================================================

-- Check 1: Uppercase emails → 
SELECT COUNT(*) AS uppercase_email_count
FROM retail_sales_catalog.silver.DimCustomer
WHERE Email != LOWER(Email);

-- Check 2: CustomerName not in Proper Case → 
SELECT COUNT(*) AS improper_name_count
FROM retail_sales_catalog.silver.DimCustomer
WHERE CustomerName != INITCAP(CustomerName);

-- Check 3: City with leading/trailing spaces → 
SELECT COUNT(*) AS city_spaces_count
FROM retail_sales_catalog.silver.DimCustomer
WHERE City != TRIM(City);

-- Check 4: Duplicate active CustomerIDs → 
SELECT COUNT(*) AS duplicate_active_customers
FROM (
    SELECT CustomerID, COUNT(*) AS cnt
    FROM retail_sales_catalog.silver.DimCustomer
    WHERE IsActive = 1
    GROUP BY CustomerID
    HAVING cnt > 1
);

-- Check 5: CustomerSK duplicate check → 
SELECT COUNT(*) AS duplicate_sk_count
FROM (
    SELECT CustomerSK, COUNT(*) AS cnt
    FROM retail_sales_catalog.silver.DimCustomer
    GROUP BY CustomerSK
    HAVING cnt > 1
);
-- ============================================================
-- SECTION 3: SCD TYPE 2 VALIDATION
-- ============================================================

-- Check 1: SCD2 Active vs Expired summary
SELECT IsActive, COUNT(*) AS count
FROM retail_sales_catalog.silver.DimCustomer
GROUP BY IsActive;

-- Check 2: Inactive records with wrong EndDate → 
SELECT COUNT(*) AS wrong_enddate_count
FROM retail_sales_catalog.silver.DimCustomer
WHERE IsActive = 0
  AND EndDate = DATE('9999-12-31');

-- Check 3: Active records without default EndDate → 
SELECT COUNT(*) AS wrong_active_enddate
FROM retail_sales_catalog.silver.DimCustomer
WHERE IsActive = 1
  AND EndDate != DATE('9999-12-31');

-- Check 4: StartDate is NULL for any record → 
SELECT COUNT(*) AS null_startdate_count
FROM retail_sales_catalog.silver.DimCustomer
WHERE StartDate IS NULL;

-- Check 5: Show full SCD2 history for changed customers
SELECT CustomerID, CustomerName, City, Address,
       StartDate, EndDate, IsActive
FROM retail_sales_catalog.silver.DimCustomer
WHERE CustomerID IN (
    SELECT CustomerID FROM retail_sales_catalog.silver.DimCustomer
    GROUP BY CustomerID HAVING COUNT(*) > 1
)
ORDER BY CustomerID, StartDate;

-- ============================================================
-- SECTION 4: DIMPRODUCT VALIDATION
-- ============================================================

-- Check 1: UnitPrice = 0 in Silver
SELECT COUNT(*) AS zero_price_count
FROM retail_sales_catalog.silver.DimProduct
WHERE UnitPrice = 0;

-- Check 2: How many zero-price products rejected from source
SELECT COUNT(*) AS rejected_from_bronze
FROM retail_sales_catalog.bronze.products_raw
WHERE UnitPrice = 0;

-- Check 3: ProductName with extra spaces → Expected: 0
SELECT COUNT(*) AS product_name_spaces
FROM retail_sales_catalog.silver.DimProduct
WHERE ProductName != TRIM(ProductName);

-- Check 4: Category with extra spaces → Expected: 0
SELECT COUNT(*) AS category_spaces
FROM retail_sales_catalog.silver.DimProduct
WHERE Category != TRIM(Category);

-- Check 5: Duplicate ProductSK → Expected: 0
SELECT COUNT(*) AS duplicate_product_sk
FROM (
    SELECT ProductSK, COUNT(*) AS cnt
    FROM retail_sales_catalog.silver.DimProduct
    GROUP BY ProductSK
    HAVING cnt > 1
);-- ============================================================
-- SECTION 5: DIMSTORE VALIDATION
-- ============================================================

-- Check 1: NULL Region in Silver → 
SELECT COUNT(*) AS null_region_count
FROM retail_sales_catalog.silver.DimStore
WHERE Region IS NULL;

-- Check 2: How many NULL regions existed in source
SELECT COUNT(*) AS null_region_in_bronze
FROM retail_sales_catalog.bronze.stores_raw
WHERE Region IS NULL OR TRIM(Region) = '';

-- Check 3: StoreName not in Proper Case → Expected: 0
SELECT COUNT(*) AS improper_storename_count
FROM retail_sales_catalog.silver.DimStore
WHERE StoreName != INITCAP(StoreName);

-- Check 4: StoreName with extra spaces → Expected: 0
SELECT COUNT(*) AS storename_spaces
FROM retail_sales_catalog.silver.DimStore
WHERE StoreName != TRIM(StoreName);

-- Check 5: Duplicate StoreSK → Expected: 0
SELECT COUNT(*) AS duplicate_store_sk
FROM (
    SELECT StoreSK, COUNT(*) AS cnt
    FROM retail_sales_catalog.silver.DimStore
    GROUP BY StoreSK
    HAVING cnt > 1
);
-- ============================================================
-- SECTION 6: FACTSALES VALIDATION
-- ============================================================

-- Check 1: Duplicate TransactionIDs in Silver → 
SELECT COUNT(*) AS duplicate_txn_count
FROM (
    SELECT TransactionID, COUNT(*) AS cnt
    FROM retail_sales_catalog.silver.FactSales
    GROUP BY TransactionID
    HAVING cnt > 1
);

-- Check 2: Zero Quantity records in Silver → 
SELECT COUNT(*) AS zero_qty_count
FROM retail_sales_catalog.silver.FactSales
WHERE Quantity = 0;

-- Check 3: NULL SKs in FactSales (orphan records) → 
SELECT COUNT(*) AS null_sk_count
FROM retail_sales_catalog.silver.FactSales
WHERE CustomerSK IS NULL
   OR ProductSK  IS NULL
   OR StoreSK    IS NULL;

-- Check 4: Amount calculation validation →
SELECT
    f.TransactionID,
    f.Quantity,
    p.UnitPrice,
    f.Amount,
    ROUND(f.Quantity * p.UnitPrice, 2) AS Expected_Amount,
    CASE
        WHEN f.Amount = ROUND(f.Quantity * p.UnitPrice, 2)
        THEN 'PASS' ELSE 'FAIL'
    END AS Amount_Check
FROM retail_sales_catalog.silver.FactSales f
JOIN retail_sales_catalog.silver.DimProduct p
    ON f.ProductSK = p.ProductSK
LIMIT 10;

-- Check 5: TxnDate format incorrect → 
SELECT COUNT(*) AS wrong_date_format
FROM retail_sales_catalog.silver.FactSales
WHERE TxnDate NOT RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}$';